# What an API actually is

**Before this notebook you need nothing except a browser and `requests`.**

An API sounds like a thing you have to be let into. It is not. It is a web
address that answers in a shape a program can use.

That is the whole idea. Everything else in this notebook is detail hanging
off it.

We use one weather service for most of the notebook, then four other
services at the end to show the same three lines of Python work everywhere.

---

## 0. Do this in your browser first

Open both of these. Same site, same city, one difference at the end.

1. <https://wx.rejusamjohn.workers.dev/auckland>
2. <https://wx.rejusamjohn.workers.dev/auckland?format=json>

The first is drawn for **you** — a chart, symbols, rounded numbers.

The second is written for a **program** — brackets, quotes, every decimal.

Neither one is more real than the other. They are the same weather, printed
two ways, by the same computer, at the same address.

> **When people say "the API", they mean the second shape.**

Come back here once you have seen both.

---

## 1. Setup

Two imports. `requests` does the talking, `pandas` we only need at the very
end.

In [1]:
import requests
from urllib.parse import urlparse

WX = "https://wx.rejusamjohn.workers.dev"

print("requests", requests.__version__)

requests 2.34.2


---

## 2. One line of Python replaces the browser

When you typed that address, your browser sent a **request** and got back a
**response**. `requests.get` does the identical thing.

In [2]:
WX = "https://wx.rejusamjohn.workers.dev"
response = requests.get(WX + "/auckland?format=json", timeout=15)

print(response)

<Response [200]>


`<Response [200]>`. That is it. You have called an API.

The `timeout=15` says: give up after 15 seconds. Always write it. Without
it, a server that never replies hangs your notebook forever, and the only
way out is restarting the kernel.

---

## 3. Every answer has two halves

**Half one: did it work?** That is the status code.

**Half two: what came back?** That is the body.

Beginners reach straight for the body and forget the status. Then they spend
twenty minutes debugging a "weird result" that was an error message all
along.

In [3]:
print("status code:", response.status_code)
print("did it work:", response.ok)
print("body size  :", len(response.content), "bytes")

status code: 200
did it work: True
body size  : 21550 bytes


The three status codes worth memorising today:

| Code | Meaning | Whose fault |
|---|---|---|
| **200** | Here you go | nobody's |
| **404** | I have no such thing | usually yours — check the address |
| **500** | I broke | theirs |

A fuller rule of thumb: **4xx means you asked wrong, 5xx means they went
wrong.**

---

## 4. The body arrives in three shapes

The same body, offered three ways. Which one you want depends on what you
are doing.

In [4]:
print("response.content ->", type(response.content))
print("response.text    ->", type(response.text))
print("response.json()  ->", type(response.json()))

response.content -> <class 'bytes'>
response.text    -> <class 'str'>
response.json()  -> <class 'dict'>


- **`.content`** is raw **bytes**. Use it for images and files.
- **`.text`** is a **str**. Use it when the answer is meant to be read.
- **`.json()`** is a **dict** (sometimes a list). Use it when the answer is
  data.

`.json()` is not magic. It reads the text and builds Python objects from it.
If the text is not JSON, it raises — you will see that happen in section 7.

Here are the first 120 characters of the text, so you can see what `.json()`
is working from:

In [5]:
print(response.text[:120])

{"location":{"name":"Auckland","country":"New Zealand","latitude":-36.84853,"longitude":174.76349,"timezone":"Pacific/Au


'{"location":{"name":"Auckland","country":"New Zealand","latitude":-36.84853,"longitude":174.76349,"timezone":"Pacific/Auckland"},"generated_at":"2026-09-16T08:18:36.351Z","data_time":"2026-09-16T08:16:25.620Z","stale":false,"units":{"temperature":"C","precipitation":"mm"},"models":["ecmwf_ifs025","gfs_seamless","icon_seamless","ukmo_seamless","jma_seamless","gem_seamless","meteofrance_seamless","cma_grapes_global"],"hourly":{"time":["2026-09-16T00:00","2026-09-16T01:00","2026-09-16T02:00","2026-09-16T03:00","2026-09-16T04:00","2026-09-16T05:00","2026-09-16T06:00","2026-09-16T07:00","2026-09-16T08:00","2026-09-16T09:00","2026-09-16T10:00","2026-09-16T11:00","2026-09-16T12:00","2026-09-16T13:00","2026-09-16T14:00","2026-09-16T15:00","2026-09-16T16:00","2026-09-16T17:00","2026-09-16T18:00","2026-09-16T19:00","2026-09-16T20:00","2026-09-16T21:00","2026-09-16T22:00","2026-09-16T23:00","2026-09-17T00:00","2026-09-17T01:00","2026-09-17T02:00","2026-09-17T03:00","2026-09-17T04:00","2026-09-17

Brackets, quotes, colons. Looks like a Python dict because JSON and Python
dicts were designed to look alike. They are not the same thing — JSON is
text, a dict is an object in memory — but the resemblance is deliberate and
it is why this is easy.

---

## 5. Once it is a dict, it is ordinary Python

No new skills needed from here. Square brackets and loops, the same as any
dictionary you have written yourself.

In [7]:
forecast = response.json()

print("the keys:", sorted(forecast))

the keys: ['attribution', 'daily', 'data_time', 'generated_at', 'hourly', 'location', 'models', 'stale', 'units']


In [8]:
forecast

{'location': {'name': 'Auckland',
  'country': 'New Zealand',
  'latitude': -36.84853,
  'longitude': 174.76349,
  'timezone': 'Pacific/Auckland'},
 'generated_at': '2026-09-16T08:18:36.351Z',
 'data_time': '2026-09-16T08:16:25.620Z',
 'stale': False,
 'units': {'temperature': 'C', 'precipitation': 'mm'},
 'models': ['ecmwf_ifs025',
  'gfs_seamless',
  'icon_seamless',
  'ukmo_seamless',
  'jma_seamless',
  'gem_seamless',
  'meteofrance_seamless',
  'cma_grapes_global'],
 'hourly': {'time': ['2026-09-16T00:00',
   '2026-09-16T01:00',
   '2026-09-16T02:00',
   '2026-09-16T03:00',
   '2026-09-16T04:00',
   '2026-09-16T05:00',
   '2026-09-16T06:00',
   '2026-09-16T07:00',
   '2026-09-16T08:00',
   '2026-09-16T09:00',
   '2026-09-16T10:00',
   '2026-09-16T11:00',
   '2026-09-16T12:00',
   '2026-09-16T13:00',
   '2026-09-16T14:00',
   '2026-09-16T15:00',
   '2026-09-16T16:00',
   '2026-09-16T17:00',
   '2026-09-16T18:00',
   '2026-09-16T19:00',
   '2026-09-16T20:00',
   '2026-09-16T21:00',

In [9]:
print("place   :", forecast["location"]["name"])
print("country :", forecast["location"]["country"])
print("timezone:", forecast["location"]["timezone"])
print("units   :", forecast["units"])

place   : Auckland
country : New Zealand
timezone: Pacific/Auckland
units   : {'temperature': 'C', 'precipitation': 'mm'}


In [10]:
forecast["location"]

{'name': 'Auckland',
 'country': 'New Zealand',
 'latitude': -36.84853,
 'longitude': 174.76349,
 'timezone': 'Pacific/Auckland'}

`forecast["location"]["name"]` is a dict inside a dict. Real API answers nest
several layers deep. Work down one level at a time and print as you go —
guessing the whole path in one line is how you end up with `KeyError`.

In [11]:
print("days returned:", len(forecast["daily"]))
print()

for day in forecast["daily"]:
    print("  {}  {:>5} to {:<5} degrees".format(
        day["date"], day["consensus_min"], day["consensus_max"]))

days returned: 7

  2026-09-16    9.8 to 15.9  degrees
  2026-09-17   10.7 to 17.7  degrees
  2026-09-18   12.3 to 15.7  degrees
  2026-09-19   12.7 to 15.7  degrees
  2026-09-20   12.2 to 15.2  degrees
  2026-09-21   11.6 to 15    degrees
  2026-09-22   12.5 to 17.4  degrees


A list of dicts, looped over with a plain `for`. That is the single most
common shape an API will hand you, and you already know how to read it.

---

## 6. A URL is four parts, and you should be able to name them

Before we change the address, take one apart. `urlparse` does it for you.

In [12]:
WX = "https://wx.rejusamjohn.workers.dev"
parts = urlparse(WX + "/auckland?format=json&days=3")

print("scheme (how to talk)     :", parts.scheme)
print("netloc (which computer)  :", parts.netloc)
print("path   (which thing)     :", parts.path)
print("query  (how you want it) :", parts.query)

scheme (how to talk)     : https
netloc (which computer)  : wx.rejusamjohn.workers.dev
path   (which thing)     : /auckland
query  (how you want it) : format=json&days=3


Say this one out loud, because everything else follows from it:

> **The path says *which thing*. The query says *how you want it*.**

`/auckland` is the thing. `?format=json` does not change the weather in
Auckland — it changes the shape it is printed in.

---

## 7. Proving it: one path, three shapes

> **Predict first.** Below we ask for the same city three times, changing
> only `format`. Will the status code change? Will the size change?

`content-type` is the server telling you what kind of body it just sent.

In [13]:
for shape in ["json", "postcard", "4"]:
    r = requests.get(WX + "/auckland", params={"format": shape}, timeout=15)
    print("{:<9} -> {}  {:<34} {:>6} bytes".format(
        shape, r.status_code, r.headers["content-type"], len(r.content)))

json      -> 200  application/json; charset=utf-8     21550 bytes
postcard  -> 200  text/plain; charset=utf-8            3001 bytes
4         -> 200  text/plain; charset=utf-8              55 bytes


Three different bodies. Three different content types. **One unchanged
`200`, one unchanged path.**

So `200` does not tell you what you got. It tells you the conversation
finished politely. Reading `content-type` before reaching for `.json()` is a
habit worth building now.

Try it yourself: `format=postcard` is a picture made of text. The extra `T`
turns the colours off — without it you get the colour codes printed raw in
some editors.

In [14]:
card = requests.get(WX + "/auckland?T", params={"format": "postcard"}, timeout=15)
print(card.url)
print(card.headers["content-type"])
print(card.text[:600])

https://wx.rejusamjohn.workers.dev/auckland?T&format=postcard
text/plain; charset=utf-8
┌──────────────────────────────────────────────────────────┐
│           :.*                       : . :     .          │
│      :              :                        :     :     │
│       :   :. :             . :        :     :     :      │
│               :                                          │
│  ::                                         :          : │
│        :    :         :                                  │
│   :                         :                   :        │
│                             .      :*      .             │
│ ▒▒▒                           ▒▒        ▒▒▒▒▒▒   


In [15]:
card = requests.get(WX + "/auckland?T", params={"format": "json"}, timeout=15)
print(card.url)
print(card.headers["content-type"])
print(card.text[:600])

https://wx.rejusamjohn.workers.dev/auckland?T&format=json
application/json; charset=utf-8
{"location":{"name":"Auckland","country":"New Zealand","latitude":-36.84853,"longitude":174.76349,"timezone":"Pacific/Auckland"},"generated_at":"2026-09-16T08:49:53.699Z","data_time":"2026-09-16T08:46:34.500Z","stale":false,"units":{"temperature":"C","precipitation":"mm"},"models":["ecmwf_ifs025","gfs_seamless","icon_seamless","ukmo_seamless","jma_seamless","gem_seamless","meteofrance_seamless","cma_grapes_global"],"hourly":{"time":["2026-09-16T00:00","2026-09-16T01:00","2026-09-16T02:00","2026-09-16T03:00","2026-09-16T04:00","2026-09-16T05:00","2026-09-16T06:00","2026-09-16T07:00","2026-09-16


---

## 8. Let `requests` build the query string

You can glue options onto the address by hand. Better: hand `requests` a
dictionary and let it assemble the string.

In [16]:
options = {"format": "json", "models": "ecmwf,gfs", "days": 2}

built = requests.get(WX + "/auckland", params=options, timeout=15)

print(built.url)

https://wx.rejusamjohn.workers.dev/auckland?format=json&models=ecmwf%2Cgfs&days=2


Look at what it did to the comma: `ecmwf,gfs` became `ecmwf%2Cgfs`. That is
**URL encoding** — some characters are not safe to put in an address raw, so
they get rewritten. The server turns them back.

Doing that by hand is where beginners lose an afternoon. Use `params`.

---

## 9. A "no" is an answer, not a crash

> **Predict first.** We ask for a city that does not exist, and for more days
> than the service will give. Does Python raise an error?

In [17]:
nowhere = requests.get(WX + "/zzzznowhere", params={"format": "json"}, timeout=15)
too_many = requests.get(WX + "/auckland", params={"format": "json", "days": 30}, timeout=15)

print("nowhere  :", nowhere.status_code, "|", nowhere.headers["content-type"])
print("too_many :", too_many.status_code, "|", too_many.headers["content-type"])
too_many.url

nowhere  : 404 | text/plain; charset=utf-8
too_many : 400 | text/plain; charset=utf-8


'https://wx.rejusamjohn.workers.dev/auckland?format=json&days=30'

**Neither one raised.** Python is happy — it asked a question and got an
answer. The answer happens to be "no".

This is the single most important thing in this notebook. `requests` does not
throw an exception because a server said no. If you do not check
`status_code` yourself, a `404` sails past and breaks something later, far
from the line that caused it.

Now read what the "no" actually said:

In [18]:
print(nowhere.text)
print("-" * 60)
print(too_many.text[:200])

Place not found: zzzznowhere
Try a nearby city, or add a country code, for example /wellington,nz

------------------------------------------------------------
Bad request: days must be a whole number from 1 to 7

Valid options: 0 1 2 3 (days), T (no colour), q (no header), n (narrow), F (no footer), u (°F), m (°C), A (force ANSI); days=1..7, models=ecmwf,gf


Both errors are written for a human to read, and both tell you how to fix it.
Good APIs do this. **Always print the body of an error** before you start
guessing.



### How to handle it?

**Check it yourself**, when you want to react differently to different codes:

```python
if response.status_code == 200:
    data = response.json()
else:
    print("no luck:", response.status_code, response.text)
```

**Or let `requests` raise**, when any failure should just stop everything:

In [19]:
if nowhere.status_code == 200:
    data = nowhere.json()
else:
    print("no luck:", nowhere.status_code, nowhere.text)

no luck: 404 Place not found: zzzznowhere
Try a nearby city, or add a country code, for example /wellington,nz



---

## 10. The same three lines work everywhere

Nothing so far was special to the weather service. Here are four completely
unrelated public APIs, none of which need a key, all answered by the same
`requests.get`.

In [20]:
services = [
    ("a cat fact",     "https://catfact.ninja/fact"),
    ("a NZ postcode",  "https://api.zippopotam.us/nz/9016"),
    ("the space station", "http://api.open-notify.org/iss-now.json"),
    ("a GitHub user",  "https://api.github.com/users/rejusam"),
]

answers = {}

for name, address in services:
    r = requests.get(address, timeout=15)
    answers[name] = r
    print("{:<20} {}  {:>5} bytes  keys: {}".format(
        name, r.status_code, len(r.content), sorted(r.json())[:4]))

a cat fact           200    108 bytes  keys: ['fact', 'length']
a NZ postcode        200    205 bytes  keys: ['country', 'country abbreviation', 'places', 'post code']
the space station    200    114 bytes  keys: ['iss_position', 'message', 'timestamp']
a GitHub user        200   1396 bytes  keys: ['avatar_url', 'bio', 'blog', 'company']


In [23]:
r.json()

{'login': 'rejusam',
 'id': 6176357,
 'node_id': 'MDQ6VXNlcjYxNzYzNTc=',
 'avatar_url': 'https://avatars.githubusercontent.com/u/6176357?v=4',
 'gravatar_id': '',
 'url': 'https://api.github.com/users/rejusam',
 'html_url': 'https://github.com/rejusam',
 'followers_url': 'https://api.github.com/users/rejusam/followers',
 'following_url': 'https://api.github.com/users/rejusam/following{/other_user}',
 'gists_url': 'https://api.github.com/users/rejusam/gists{/gist_id}',
 'starred_url': 'https://api.github.com/users/rejusam/starred{/owner}{/repo}',
 'subscriptions_url': 'https://api.github.com/users/rejusam/subscriptions',
 'organizations_url': 'https://api.github.com/users/rejusam/orgs',
 'repos_url': 'https://api.github.com/users/rejusam/repos',
 'events_url': 'https://api.github.com/users/rejusam/events{/privacy}',
 'received_events_url': 'https://api.github.com/users/rejusam/received_events',
 'type': 'User',
 'user_view_type': 'public',
 'site_admin': False,
 'name': 'Reju Sam John',

Four organisations who have never spoken to each other, and one loop reads
all of them. **That is what "API" buys you** — an agreement about shape, so
the same code works against things it was not written for.

Now pull one value out of each. Different keys, identical technique:

In [ ]:
print("cat fact :", answers["a cat fact"].json()["fact"])
print()

postcode = answers["a NZ postcode"].json()
print("postcode :", postcode["post code"], "is", postcode["places"][0]["place name"])
print()

iss = answers["the space station"].json()
print("ISS is at:", iss["iss_position"]["latitude"], iss["iss_position"]["longitude"])
print()

user = answers["a GitHub user"].json()
print("GitHub   :", user["login"], "joined", user["created_at"], "-", user["public_repos"], "public repos")

> **One warning about `api.open-notify.org`.** Look at its address: `http`,
> not `https`. There is no lock on it. Anything sent to it travels in the
> clear and could be read or altered on the way. Fine for the position of the
> space station. Never for anything you would mind a stranger reading.

---

## 11. Four manners that keep you welcome

1. **Always pass `timeout=`.** Your notebook should never hang on someone
   else's slow server.
2. **Ask once, reuse the answer.** Store the response in a variable. A loop
   that calls the same address thirty times is thirty requests, and every one
   costs the other side money.
3. **Read the limit if there is one.** If `x-ratelimit-remaining` is there,
   look at it.
4. **Credit the source.** This service ships its own attribution, because the
   data underneath is licensed:

In [27]:
response = requests.get("https://api.github.com/users/rejusam")

# Print the rate limit information
print("Total Limit:", response.headers.get("X-RateLimit-Limit"))
print("Remaining:", response.headers.get("X-RateLimit-Remaining"))
print("Resets at (epoch):", response.headers.get("X-RateLimit-Reset"))

Total Limit: 60
Remaining: 55
Resets at (epoch): 1789553310


In [28]:
import datetime

reset_epoch = int(response.headers.get("X-RateLimit-Reset"))
reset_time = datetime.datetime.fromtimestamp(reset_epoch)

print("Limit resets at:", reset_time.strftime('%I:%M %p'))

Limit resets at: 10:08 PM


---

## What to take away

1. **An API is a web address that answers in a shape a program can use.** The
   browser page and the JSON are the same weather, printed differently.
2. **Two halves: status and body.** Check the status before you trust the
   body.
3. **A `404` does not raise.** If you do not check, it slips past and breaks
   something later.
4. **Path = which thing, query = how you want it.** Pass the query as a
   `params` dictionary and let `requests` encode it.
5. **`.json()` gives you a plain dict.** From there it is ordinary Python —
   brackets and loops, no new skills.
6. **Print the keys of anything new** before writing code against it.
7. **Headers talk about the answer** — the content type, how long it stays
   fresh, and sometimes how much of your allowance is left.
8. **Everything is somebody's client.** The service you call is probably
   calling another one.

### Try these yourself

- Swap `auckland` for your own town. 
- Ask the weather service for `format=2`, `format=3`, `format=4`. What
  changes?
- Find your own postcode on `api.zippopotam.us`. The country code goes first.
- Call the GitHub API for your own username and print your repo count.

In [31]:
response = requests.get("https://wx.rejusamjohn.workers.dev/wellington?format=3")
# Print the text content of the response
print(response.text)

Wellington, New Zealand: ⛅ 12°C



In [32]:
import requests

# Querying the Zippopotam API for Mount Wellington, Auckland
response = requests.get("https://api.zippopotam.us/nz/1060")

# Parse and print the JSON data
data = response.json()
print(data)

{'country': 'New Zealand', 'country abbreviation': 'NZ', 'post code': '1060', 'places': [{'place name': 'Penrose', 'longitude': '174.8356', 'latitude': '-36.9177', 'state': '', 'state abbreviation': ''}]}


In [33]:
import requests

# Querying the GitHub API for your profile
response = requests.get("https://api.github.com/users/rejusam")

# Parse the JSON response
data = response.json()

# Extract and print the repository count
print("Public Repositories:", data["public_repos"])

Public Repositories: 31


# Reading pages that were never meant for programs

Just ended on a comfortable idea: ask an API, get a dict.

Most of the web is not like that. Most of the web is **HTML written for a
human eye** — no `format=json`, no documented keys, no promises.

**BeautifulSoup** reads that HTML as a tree you can search. Install it as
`beautifulsoup4`, import it as `bs4` — the names differ, and it catches
everyone once.

Below, a bookshop built specifically for scraping practice. Two search
methods do all the work:

- **`find`** — the first matching tag
- **`find_all`** — a list of every match, which an ordinary `for` loop reads

You find the tag and class names by opening the page in your browser,
right-clicking a price, and choosing **Inspect**. That part is not Python.

You need one new library:

```
pip install bs4
```

The package is `beautifulsoup4`, the import is `bs4`.
It catches everyone once.

In [34]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# WX = "https://wx.rejusamjohn.workers.dev"

# print("bs4 is installed and imported")

In [37]:
from bs4 import BeautifulSoup

page = requests.get("https://books.toscrape.com/", timeout=15)
page.encoding = "utf-8"        # the server forgot to say which alphabet it used

soup = BeautifulSoup(page.text, "html.parser")

print("books found on this page:", len(soup.find_all("article", class_="product_pod")))
print()

for card in soup.find_all("article", class_="product_pod")[:5]:
    title = card.h3.a["title"]                                 # a tag's attribute
    price = card.find("p", class_="price_color").get_text()    # a tag's text
    print("{:<40} {}".format(title[:38], price))

books found on this page: 20

A Light in the Attic                     £51.77
Tipping the Velvet                       £53.74
Soumission                               £50.10
Sharp Objects                            £47.82
Sapiens: A Brief History of Humankind    £54.23


In [36]:
soup

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s